In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, fbeta_score,
    confusion_matrix, classification_report
)

# =========================
# 1) Daten laden + Mini-Clean
# =========================
# Erklärung: CSV einlesen
df = pd.read_csv("Marktkampagne.csv")

# Erklärung: Datum parsen (TT-MM-JJJJ)
df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], dayfirst=True, errors="raise")

# Erklärung: Einkommen-Fehlwerte (Flag + Median)
df["Einkommen_fehlte"] = df["Einkommen"].isna().astype(int)
df["Einkommen"] = df["Einkommen"].fillna(df["Einkommen"].median())

# Erklärung: Kundendauer als Feature
stichtag = df["Datum_Kunde"].max()
df["Kundendauer_Tage"] = (stichtag - df["Datum_Kunde"]).dt.days

# =========================
# 2) Churn-Target bauen
# =========================
# Erklärung: Churn-Risiko = Top-10% der höchsten "Letzter_Kauf_Tage"
churn_thr = df["Letzter_Kauf_Tage"].quantile(0.90)
df["Ziel_ChurnRisiko"] = (df["Letzter_Kauf_Tage"] >= churn_thr).astype(int)

print("Churn-Schwelle (Letzter_Kauf_Tage):", float(churn_thr))
print("Churn-Rate (Anteil 1):", df["Ziel_ChurnRisiko"].mean())

# =========================
# 3) Features bauen (Leakage vermeiden!)
# =========================
# Erklärung: Letzter_Kauf_Tage raus, weil daraus das Target definiert wurde
ziel = "Ziel_ChurnRisiko"
drop_cols = ["ID", "Datum_Kunde", ziel, "Letzter_Kauf_Tage"]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[ziel]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# Erklärung: Train/Test Split (stratify hält Klassenverteilung stabil)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train-Rate:", y_train.mean(), "| Test-Rate:", y_test.mean())

# =========================
# 4) Modell (Baseline) + Scores
# =========================
# Erklärung: Preprocessing für Zahlen/Kategorien
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# Erklärung: Logistic Regression + class_weight=balanced (gegen "immer 0" Problem)
model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]

print("\nROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC:", average_precision_score(y_test, proba))

# =========================
# 5) Threshold-Optimierung (F2 -> Recall wichtiger)
# =========================
# Erklärung: Wir suchen die Schwelle, die F2 maximiert (Churn: lieber weniger verpassen)
thresholds = np.linspace(0.05, 0.95, 19)

best = {"t": None, "precision": None, "recall": None, "f2": -1, "positives_pred": None}
for t in thresholds:
    pred = (proba >= t).astype(int)
    p = precision_score(y_test, pred, zero_division=0)
    r = recall_score(y_test, pred, zero_division=0)
    f2 = fbeta_score(y_test, pred, beta=2, zero_division=0)

    if f2 > best["f2"]:
        best = {"t": float(t), "precision": p, "recall": r, "f2": f2, "positives_pred": int(pred.sum())}

pred_best = (proba >= best["t"]).astype(int)

print("\nBeste Schwelle (F2):", best["t"])
print({"precision": best["precision"], "recall": best["recall"], "f2": best["f2"], "positives_pred": best["positives_pred"]})
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_best))
print("\nReport:\n", classification_report(y_test, pred_best))



Churn-Schwelle (Letzter_Kauf_Tage): 89.0
Churn-Rate (Anteil 1): 0.10625
Train-Rate: 0.10602678571428571 | Test-Rate: 0.10714285714285714

ROC-AUC: 0.5015625
PR-AUC: 0.10623558587044257

Beste Schwelle (F2): 0.2
{'precision': 0.11483253588516747, 'recall': 1.0, 'f2': 0.39344262295081966, 'positives_pred': 418}
Confusion Matrix:
 [[ 30 370]
 [  0  48]]

Report:
               precision    recall  f1-score   support

           0       1.00      0.07      0.14       400
           1       0.11      1.00      0.21        48

    accuracy                           0.17       448
   macro avg       0.56      0.54      0.17       448
weighted avg       0.91      0.17      0.15       448



In [ ]:
# Erklärung: Unser "Churn" ist direkt über Recency definiert (Letzter_Kauf_Tage).
# Wenn wir Letzter_Kauf_Tage als Feature entfernen (Leakage vermeiden), bleibt kaum Signal übrig:
# -> ROC-AUC ~ 0.5 / PR-AUC ~ Grundrate => Modell rät praktisch.
# Wenn wir Letzter_Kauf_Tage drin lassen, ist das Modell "zu gut", weil es genau die Ziel-Info enthält:
# -> ML wäre nur eine komplizierte Version einer einfachen Regel.
# Fazit: Für dieses Label ist eine Recency-Regel transparenter, stabiler und sinnvoller als ML.
